In [1]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers

import numpy as np
import tensorflow as tf
import gymnasium as gym
import scipy.signal

## Functions and class

In [2]:
def discounted_cumulative_sums(x, discount):
    # Discounted cumulative sums of vectors for computing rewards-to-go and advantage estimates
    return scipy.signal.lfilter([1], [1, float(-discount)], x[::-1], axis=0)[::-1]


class Buffer:
    # Buffer for storing trajectories
    def __init__(self, observation_dimensions, size, gamma=0.99, lam=0.95):
        # Buffer initialization
        self.observation_buffer = np.zeros(
            (size, observation_dimensions), dtype=np.float32
        )
        self.action_buffer = np.zeros(size, dtype=np.int32)
        self.advantage_buffer = np.zeros(size, dtype=np.float32)
        self.reward_buffer = np.zeros(size, dtype=np.float32)
        self.return_buffer = np.zeros(size, dtype=np.float32)
        self.value_buffer = np.zeros(size, dtype=np.float32)
        self.logprobability_buffer = np.zeros(size, dtype=np.float32)
        self.gamma, self.lam = gamma, lam
        self.pointer, self.trajectory_start_index = 0, 0

    def store(self, observation, action, reward, value, logprobability):
        # Append one step of agent-environment interaction
        self.observation_buffer[self.pointer] = observation
        self.action_buffer[self.pointer] = action
        self.reward_buffer[self.pointer] = reward
        self.value_buffer[self.pointer] = value
        self.logprobability_buffer[self.pointer] = logprobability
        self.pointer += 1

    def finish_trajectory(self, last_value=0):
        # Finish the trajectory by computing advantage estimates and rewards-to-go
        path_slice = slice(self.trajectory_start_index, self.pointer)
        rewards = np.append(self.reward_buffer[path_slice], last_value)
        values = np.append(self.value_buffer[path_slice], last_value)

        deltas = rewards[:-1] + self.gamma * values[1:] - values[:-1]

        self.advantage_buffer[path_slice] = discounted_cumulative_sums(
            deltas, self.gamma * self.lam
        )
        self.return_buffer[path_slice] = discounted_cumulative_sums(
            rewards, self.gamma
        )[:-1]

        self.trajectory_start_index = self.pointer

    def get(self):
        # Get all data of the buffer and normalize the advantages
        self.pointer, self.trajectory_start_index = 0, 0
        advantage_mean, advantage_std = (
            np.mean(self.advantage_buffer),
            np.std(self.advantage_buffer),
        )
        self.advantage_buffer = (self.advantage_buffer - advantage_mean) / advantage_std
        return (
            self.observation_buffer,
            self.action_buffer,
            self.advantage_buffer,
            self.return_buffer,
            self.logprobability_buffer,
        )


def mlp(x, sizes, activation=keras.activations.tanh, output_activation=None):
    # Build a feedforward neural network
    for size in sizes[:-1]:
        x = layers.Dense(units=size, activation=activation)(x)
    return layers.Dense(units=sizes[-1], activation=output_activation)(x)


def logprobabilities(logits, a):
    # Compute the log-probabilities of taking actions a by using the logits (i.e. the output of the actor)
    logprobabilities_all = keras.ops.log_softmax(logits)
    logprobability = keras.ops.sum(
        keras.ops.one_hot(a, num_actions) * logprobabilities_all, axis=1
    )
    return logprobability


seed_generator = keras.random.SeedGenerator(1337)


# Sample action from actor
@tf.function
def sample_action(observation):
    logits = actor(observation)
    action = keras.ops.squeeze(
        keras.random.categorical(logits, 1, seed=seed_generator), axis=1
    )
    return logits, action


# Train the policy by maxizing the PPO-Clip objective
@tf.function
def train_policy(
    observation_buffer, action_buffer, logprobability_buffer, advantage_buffer
):
    with tf.GradientTape() as tape:  # Record operations for automatic differentiation.
        ratio = keras.ops.exp(
            logprobabilities(actor(observation_buffer), action_buffer)
            - logprobability_buffer
        )
        min_advantage = keras.ops.where(
            advantage_buffer > 0,
            (1 + clip_ratio) * advantage_buffer,
            (1 - clip_ratio) * advantage_buffer,
        )

        policy_loss = -keras.ops.mean(
            keras.ops.minimum(ratio * advantage_buffer, min_advantage)
        )
    policy_grads = tape.gradient(policy_loss, actor.trainable_variables)
    policy_optimizer.apply_gradients(zip(policy_grads, actor.trainable_variables))

    kl = keras.ops.mean(
        logprobability_buffer
        - logprobabilities(actor(observation_buffer), action_buffer)
    )
    kl = keras.ops.sum(kl)
    return kl


# Train the value function by regression on mean-squared error
@tf.function
def train_value_function(observation_buffer, return_buffer):
    with tf.GradientTape() as tape:  # Record operations for automatic differentiation.
        value_loss = keras.ops.mean((return_buffer - critic(observation_buffer)) ** 2)
    value_grads = tape.gradient(value_loss, critic.trainable_variables)
    value_optimizer.apply_gradients(zip(value_grads, critic.trainable_variables))

## Hyperparameters

In [3]:
# Hyperparameters of the PPO algorithm
steps_per_epoch = 4000
epochs = 30
gamma = 0.99
clip_ratio = 0.2
policy_learning_rate = 3e-4
value_function_learning_rate = 1e-3
train_policy_iterations = 80
train_value_iterations = 80
lam = 0.97
target_kl = 0.01
hidden_sizes = (64, 64)

# True if you want to render the environment
render = False

In [4]:
# Experiment 1: Reduce Training Epochs
epochs = 5

In [7]:
# Experiment 2: Increase Hidden Layer Size
hidden_sizes = (128, 128)

In [11]:
# Experiment 3: Increase Clip Ratio
clip_ratio = 0.4
epochs = 30
hidden_sizes = (64, 64)

## Initializations

In [12]:
# Initialize the environment and get the dimensionality of the
# observation space and the number of possible actions
env = gym.make("CartPole-v1")
observation_dimensions = env.observation_space.shape[0]
num_actions = env.action_space.n

# Initialize the buffer
buffer = Buffer(observation_dimensions, steps_per_epoch)

# Initialize the actor and the critic as keras models
observation_input = keras.Input(shape=(observation_dimensions,), dtype="float32")
logits = mlp(observation_input, list(hidden_sizes) + [num_actions])
actor = keras.Model(inputs=observation_input, outputs=logits)
value = keras.ops.squeeze(mlp(observation_input, list(hidden_sizes) + [1]), axis=1)
critic = keras.Model(inputs=observation_input, outputs=value)

# Initialize the policy and the value function optimizers
policy_optimizer = keras.optimizers.Adam(learning_rate=policy_learning_rate)
value_optimizer = keras.optimizers.Adam(learning_rate=value_function_learning_rate)

# Initialize the observation, episode return and episode length
observation, _ = env.reset()
episode_return, episode_length = 0, 0

## Train

In [13]:
# Iterate over the number of epochs
for epoch in range(epochs):
    # Initialize the sum of the returns, lengths and number of episodes for each epoch
    sum_return = 0
    sum_length = 0
    num_episodes = 0
    episode_rewards = []

    # Iterate over the steps of each epoch
    for t in range(steps_per_epoch):
        if render:
            env.render()

        # Get the logits, action, and take one step in the environment
        observation = observation.reshape(1, -1)
        logits, action = sample_action(observation)
        observation_new, reward, done, _, _ = env.step(action[0].numpy())
        episode_return += reward
        episode_length += 1

        # Get the value and log-probability of the action
        value_t = critic(observation)
        logprobability_t = logprobabilities(logits, action)

        # Store obs, act, rew, v_t, logp_pi_t
        buffer.store(observation, action, reward, value_t, logprobability_t)

        # Update the observation
        observation = observation_new

        # Finish trajectory if reached to a terminal state
        terminal = done
        if terminal or (t == steps_per_epoch - 1):
            last_value = 0 if done else critic(observation.reshape(1, -1))
            buffer.finish_trajectory(last_value)
            sum_return += episode_return
            sum_length += episode_length
            num_episodes += 1
            episode_rewards.append(episode_return)
            observation, _ = env.reset()
            episode_return, episode_length = 0, 0

    # Get values from the buffer
    (
        observation_buffer,
        action_buffer,
        advantage_buffer,
        return_buffer,
        logprobability_buffer,
    ) = buffer.get()

    # Update the policy and implement early stopping using KL divergence
    for _ in range(train_policy_iterations):
        kl = train_policy(
            observation_buffer, action_buffer, logprobability_buffer, advantage_buffer
        )
        if kl > 1.5 * target_kl:
            # Early Stopping
            break

    # Update the value function
    for _ in range(train_value_iterations):
        train_value_function(observation_buffer, return_buffer)

    # Print mean return and length for each epoch
    print(
        f" Epoch: {epoch + 1}. Mean Return: {sum_return / num_episodes}. Mean Length: {sum_length / num_episodes}"
    )

 Epoch: 1. Mean Return: 333.3333333333333. Mean Length: 333.3333333333333
 Epoch: 2. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 3. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 4. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 5. Mean Return: 266.6666666666667. Mean Length: 266.6666666666667
 Epoch: 6. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 7. Mean Return: 200.0. Mean Length: 200.0
 Epoch: 8. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 9. Mean Return: 333.3333333333333. Mean Length: 333.3333333333333
 Epoch: 10. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 11. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 12. Mean Return: 444.44444444444446. Mean Length: 444.44444444444446
 Epoch: 13. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 14. Mean Return: 363.6363636363636. Mean Length:

In [ ]:
import matplotlib.pyplot as plt

# Example visualization of collected rewards (using the episode_rewards list from above)
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards)
plt.title("PPO Agent Rewards per Episode (Evaluation)")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(True)
plt.show()

In [ ]:
'''
Definition: The "mean return" is the expected value of the cumulative reward, averaged over many trajectories (episodes).
Purpose: It measures how "good" a policy is — higher returns indicate the agent is learning to achieve its goals effectively.

Definition: The "mean length" is the average number of steps per episode across many trials.
Purpose: This metric helps determine how quickly an agent solves a task (shorter is better for efficiency, higher is better for survival usecases) or if it is failing to reach a terminal state (too long).

1. Training outcome with Original Settings of hyperparameters

 Epoch: 1. Mean Return: 19.704433497536947. Mean Length: 19.704433497536947
 Epoch: 2. Mean Return: 21.978021978021978. Mean Length: 21.978021978021978
 Epoch: 3. Mean Return: 29.62962962962963. Mean Length: 29.62962962962963
 Epoch: 4. Mean Return: 37.735849056603776. Mean Length: 37.735849056603776
 Epoch: 5. Mean Return: 63.492063492063494. Mean Length: 63.492063492063494
 Epoch: 6. Mean Return: 88.88888888888889. Mean Length: 88.88888888888889
 Epoch: 7. Mean Return: 125.0. Mean Length: 125.0
 Epoch: 8. Mean Return: 125.0. Mean Length: 125.0
 Epoch: 9. Mean Return: 142.85714285714286. Mean Length: 142.85714285714286
 Epoch: 10. Mean Return: 250.0. Mean Length: 250.0
 Epoch: 11. Mean Return: 500.0. Mean Length: 500.0
 Epoch: 12. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 13. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 14. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 15. Mean Return: 2000.0. Mean Length: 2000.0
 Epoch: 16. Mean Return: 1000.0. Mean Length: 1000.0
 Epoch: 17. Mean Return: 1000.0. Mean Length: 1000.0
 Epoch: 18. Mean Return: 2000.0. Mean Length: 2000.0
 Epoch: 19. Mean Return: 1000.0. Mean Length: 1000.0
 Epoch: 20. Mean Return: 800.0. Mean Length: 800.0
 Epoch: 21. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 22. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 23. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 24. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 25. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 26. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 27. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 28. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 29. Mean Return: 4000.0. Mean Length: 4000.0
 Epoch: 30. Mean Return: 4000.0. Mean Length: 4000.0

 Analysis - In reinforcement learning, if both the mean return (total accumulated reward) and mean episode length (number of steps per episode) are increasing together,
 it generally indicates that the agent is learning to survive longer and explore more of the environment.


 Experiment 1 - Epochs set to 5
 Epoch: 1. Mean Return: 21.50537634408602. Mean Length: 21.50537634408602
 Epoch: 2. Mean Return: 30.76923076923077. Mean Length: 30.76923076923077
 Epoch: 3. Mean Return: 39.603960396039604. Mean Length: 39.603960396039604
 Epoch: 4. Mean Return: 52.63157894736842. Mean Length: 52.63157894736842
 Epoch: 5. Mean Return: 90.9090909090909. Mean Length: 90.9090909090909

 Analysis (5 epochs) - As the mean return is same as mean length and both increasing at the same rate the agent seems to be learning to survive longer and explore more of the environment.

 Experiment 2 - Hidden Layer Size set to (128, 128) and epochs set to 5
 Epoch: 1. Mean Return: 86.95652173913044. Mean Length: 86.95652173913044
 Epoch: 2. Mean Return: 129.03225806451613. Mean Length: 129.03225806451613
 Epoch: 3. Mean Return: 125.0. Mean Length: 125.0
 Epoch: 4. Mean Return: 181.8181818181818. Mean Length: 181.8181818181818
 Epoch: 5. Mean Return: 235.2941176470588. Mean Length: 235.2941176470588
 Analysis - As the mean return and mean length both increase significantly across the epochs, it indicates that increasing the hidden layer size to (128, 128) had a positive impact on the agent's learning,
 leading to better performance in the environment. The agent is able to survive longer and achieve higher cumulative rewards within the same number of epochs compared to Experiment 1.

 Experiment 3 - Clip Ratio set to 0.4 and epochs set to 30 (hidden_sizes remains (64, 64))
 Epoch: 1. Mean Return: 333.3333333333333. Mean Length: 333.3333333333333
 Epoch: 2. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 3. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 4. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 5. Mean Return: 266.6666666666667. Mean Length: 266.6666666666667
 Epoch: 6. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 7. Mean Return: 200.0. Mean Length: 200.0
 Epoch: 8. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 9. Mean Return: 333.3333333333333. Mean Length: 333.3333333333333
 Epoch: 10. Mean Return: 285.7142857142857. Mean Length: 285.7142857142857
 Epoch: 11. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 12. Mean Return: 444.44444444444446. Mean Length: 444.44444444444446
 Epoch: 13. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 14. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 15. Mean Return: 363.6363636363636. Mean Length: 363.6363636363636
 Epoch: 16. Mean Return: 400.0. Mean Length: 400.0
 Epoch: 17. Mean Return: 333.3333333333333. Mean Length: 333.3333333333333
 Epoch: 18. Mean Return: 235.2941176470588. Mean Length: 235.2941176470588
 Epoch: 19. Mean Return: 266.6666666666667. Mean Length: 266.6666666666667
 Epoch: 20. Mean Return: 266.6666666666667. Mean Length: 266.6666666666667
 Epoch: 21. Mean Return: 800.0. Mean Length: 800.0
 Epoch: 22. Mean Return: 444.44444444444446. Mean Length: 444.44444444444446
 Epoch: 23. Mean Return: 666.6666666666666. Mean Length: 666.6666666666666
 Epoch: 24. Mean Return: 307.6923076923077. Mean Length: 307.6923076923077
 Epoch: 25. Mean Return: 571.4285714285714. Mean Length: 571.4285714285714
 Epoch: 26. Mean Return: 2000.0. Mean Length: 2000.0
 Epoch: 27. Mean Return: 666.6666666666666. Mean Length: 666.6666666666666
 Epoch: 28. Mean Return: 1000.0. Mean Length: 1000.0
 Epoch: 29. Mean Return: 500.0. Mean Length: 500.0
 Epoch: 30. Mean Return: 1000.0. Mean Length: 1000.0

 Analysis - In Experiment 3, where the `clip_ratio` was increased to 0.4 while `epochs` were set to 30 and `hidden_sizes` back to (64, 64), the agent shows
 some fluctuations in performance, but ultimately achieves a mean return and mean length of 1000.0 by the 30th epoch. Compared to the original settings,
 the performance appears to be lower in some intermediate epochs but reaches a respectable level by the end. The higher clip ratio might allow for larger policy updates,
 which could lead to faster learning but also potentially more instability, explaining the variations seen. However, it does not reach the very high returns observed in
 the original settings (4000.0) where the agent perfectly solves the environment.
'''

## Visualizations

Before training:

![Imgur](https://i.imgur.com/rKXDoMC.gif)

After 8 epochs of training:

![Imgur](https://i.imgur.com/M0FbhF0.gif)

After 20 epochs of training:

![Imgur](https://i.imgur.com/tKhTEaF.gif)